In [5]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv("movies-mod-sim-neu.csv")  # <-- change path here if needed

df.columns = df.columns.str.strip()
df = df[["Film", "Genre", "Lead Studio"]].dropna().reset_index(drop=True)

# Merge spaces in Lead Studio so "Warner Bros" → "WarnerBros" (one token)
df["Lead Studio"] = df["Lead Studio"].str.strip().str.replace(r"\s+", "", regex=True)

# Merge spaces in Film title so "Spiderman 2" → "Spiderman2" (one token)
df["Film_token"] = df["Film"].str.strip().str.replace(r"\s+", "", regex=True)

# Combine Film + Genre + Lead Studio into one feature string
df["features"] = df["Film_token"] + " " + df["Genre"].str.strip() + " " + df["Lead Studio"]

cv = CountVectorizer(binary=True)
sim_matrix = cosine_similarity(cv.fit_transform(df["features"]))

def recommend(movie_name, n):
    matches = df[df["Film"].str.strip().str.lower() == movie_name.strip().lower()]
    if matches.empty:
        print(f"Movie '{movie_name}' not found in dataset.")
        return
    idx = matches.index[0]
    scores = sorted([(i, s) for i, s in enumerate(sim_matrix[idx]) if i != idx],
                    key=lambda x: x[1], reverse=True)
    print(f"\nTop {n} movies similar to '{df.loc[idx, 'Film']}':")
    for rank, (i, score) in enumerate(scores[:n], 1):
        print(f"  {rank}. {df.loc[i, 'Film']} (similarity: {score:.4f})")

if __name__ == "__main__":
    while True:
        name = input("\nEnter movie name (or 'quit'): ").strip()
        if name.lower() == "quit":
            break
        try:
            n = int(input("Enter number of recommendations: "))
        except ValueError:
            print("Please enter a valid integer.")
            continue
        recommend(name, n)


Top 3 movies similar to 'Spider-Man':
  1. Iron Man 3 (similarity: 0.7303)
  2. Spider-Man 2 (similarity: 0.6124)
  3. Infinite (similarity: 0.5477)

Top 3 movies similar to 'Spider-Man':
  1. Iron Man 3 (similarity: 0.7303)
  2. Spider-Man 2 (similarity: 0.6124)
  3. Infinite (similarity: 0.5477)
